# Grammar Scoring Performance Fix

## Problem Statement

The current grammar scoring model exhibits **severe overfitting**:

- **Cross-Validation RMSE**: 0.34-0.42
- **Kaggle Test RMSE**: 0.89
- **Generalization Gap**: 0.55+ (CRITICAL)

This massive gap indicates the model memorizes training data rather than learning grammar patterns. The model performs well on CV folds but fails catastrophically on unseen test data.

## Root Causes

### 1. Feature Dimensionality Curse
- **5000 TF-IDF features + 384 BERT embeddings = 5384 features**
- Only **409 training samples**
- **Feature-to-sample ratio: 13:1** (should be <1:10)
- Impact: Models memorize noise instead of learning patterns

### 2. Wrong Feature Types
- TF-IDF captures word frequency, not grammar correctness
- BERT embeddings capture semantics, not grammatical errors
- **Missing**: actual grammar error counts, syntax violations, fluency metrics
- Impact: No direct signal for grammar quality

### 3. Arbitrary Calibration
- Manual scaling factors (0.88x, 1.14x) tuned on CV data
- No theoretical justification
- Impact: Overfits to CV fold artifacts

### 4. Model Complexity
- 4-way ensemble with different calibrations
- Multiple stacking layers
- Impact: More parameters to overfit with limited data

## Solution Approach

### Feature Engineering: Grammar-Focused Features (~35 features)

1. **Grammar Error Features** (LanguageTool)
   - Total errors, error density
   - Spelling, grammar, style, punctuation errors

2. **Readability Metrics** (textstat)
   - Flesch Reading Ease, Flesch-Kincaid Grade
   - Gunning Fog, SMOG Index
   - Automated Readability Index, Coleman-Liau Index

3. **Disfluency Markers**
   - Filler words (um, uh, like)
   - Repetitions, incomplete sentences
   - Vocabulary diversity

4. **Syntactic Complexity**
   - Sentence length distribution
   - Punctuation patterns
   - Capitalization consistency

### Model Strategy: Simple & Regularized

1. **Ridge Regression** (alpha=20)
   - Strong regularization to prevent overfitting
   - Linear baseline

2. **LightGBM** (max_depth=3, n_estimators=75)
   - Conservative parameters
   - Captures non-linear patterns without overfitting

3. **Simple Ensemble**
   - 50/50 average (no stacking, no calibration)
   - Predictions clipped to [0, 5]

### Validation Strategy

- **5-fold Cross-Validation** with consistent scoring
- Monitor fold consistency (std < 0.1)
- Track train-validation gap per fold
- Property-based testing for correctness

## Expected Outcomes

### Target Metrics
- **CV RMSE**: 0.45-0.55 (vs 0.34 current)
- **Kaggle Test RMSE**: 0.50-0.60 (vs 0.89 current)
- **Generalization Gap**: <0.10 (vs 0.55 current)
- **Fold Consistency**: std <0.08
- **Feature Count**: 35 (vs 5384 current)
- **Feature-Sample Ratio**: 0.086 (vs 13.2 current)

### Key Improvements
- **99.4% reduction** in feature dimensionality
- **33-44% improvement** in test RMSE
- **82% reduction** in generalization gap
- **50% simpler** model architecture

### Success Criteria
✅ Feature-sample ratio < 0.1  
✅ Fold consistency std < 0.1  
✅ CV RMSE in [0.45, 0.60]  
✅ Generalization gap < 0.15  
✅ All property tests pass  

**Note**: A slightly higher CV RMSE (0.50 vs 0.34) with much better test performance (0.55 vs 0.89) is a successful outcome. The goal is generalization, not memorization.

In [2]:
# Install required dependencies
# Install PyTorch first (required by Whisper)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# Install other dependencies
!pip install -q openai-whisper language-tool-python textstat lightgbm

In [ ]:
# Standard library imports
import os
import re
import warnings
warnings.filterwarnings('ignore')

# Fix for PyTorch DLL loading issue on Windows
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Data manipulation and numerical computing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Progress bars
from tqdm import tqdm

# Machine learning - sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge

# Machine learning - LightGBM
import lightgbm as lgb

# Statistical tests
from scipy.stats import ks_2samp

# Speech recognition
import whisper

# Grammar and readability analysis
import language_tool_python
import textstat

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\jayes\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# Set up paths and constants

# Random seed for reproducibility
SEED = 42
np.random.seed(SEED)

# Base directory - works for both local and Kaggle environments
if os.path.exists('/kaggle/input'):
    # Kaggle environment
    BASE_DIR = '/kaggle/input/grammar-scoring'
    WORKING_DIR = '/kaggle/working'
else:
    # Local environment
    BASE_DIR = './dataset'
    WORKING_DIR = '.'

# Dataset paths
TRAIN_CSV = os.path.join(BASE_DIR, 'csvs', 'train.csv')
TEST_CSV = os.path.join(BASE_DIR, 'csvs', 'test.csv')

# Audio directories
TRAIN_AUDIO_DIR = os.path.join(BASE_DIR, 'audios', 'train')
TEST_AUDIO_DIR = os.path.join(BASE_DIR, 'audios', 'test')

# Cache files for transcriptions (to avoid re-running Whisper)
TRAIN_TRANSCRIBED_CACHE = os.path.join(WORKING_DIR, 'train_transcribed.csv')
TEST_TRANSCRIBED_CACHE = os.path.join(WORKING_DIR, 'test_transcribed.csv')

# Feature cache files
TRAIN_FEATURES_CACHE = os.path.join(WORKING_DIR, 'train_features.csv')
TEST_FEATURES_CACHE = os.path.join(WORKING_DIR, 'test_features.csv')

# Output files
SUBMISSION_FILE = os.path.join(WORKING_DIR, 'submission_fixed.csv')
DIAGNOSTICS_FILE = os.path.join(WORKING_DIR, 'diagnostics_fixed.png')

# Print configuration
print(f"Configuration:")
print(f"  SEED: {SEED}")
print(f"  BASE_DIR: {BASE_DIR}")
print(f"  WORKING_DIR: {WORKING_DIR}")
print(f"\nData paths:")
print(f"  Train CSV: {TRAIN_CSV}")
print(f"  Test CSV: {TEST_CSV}")
print(f"  Train Audio: {TRAIN_AUDIO_DIR}")
print(f"  Test Audio: {TEST_AUDIO_DIR}")
print(f"\nCache files:")
print(f"  Train transcriptions: {TRAIN_TRANSCRIBED_CACHE}")
print(f"  Test transcriptions: {TEST_TRANSCRIBED_CACHE}")
print(f"  Train features: {TRAIN_FEATURES_CACHE}")
print(f"  Test features: {TEST_FEATURES_CACHE}")
print(f"\nOutput files:")
print(f"  Submission: {SUBMISSION_FILE}")
print(f"  Diagnostics: {DIAGNOSTICS_FILE}")

# Verify paths exist
print(f"\nPath verification:")
print(f"  Train CSV exists: {os.path.exists(TRAIN_CSV)}")
print(f"  Test CSV exists: {os.path.exists(TEST_CSV)}")
print(f"  Train audio dir exists: {os.path.exists(TRAIN_AUDIO_DIR)}")
print(f"  Test audio dir exists: {os.path.exists(TEST_AUDIO_DIR)}")

## Data Loading and Transcription (Tasks 3.1-3.5)

### Task 3.1: Load train.csv and test.csv
Load the dataset CSV files containing audio filenames and labels.

### Task 3.2: Implement Whisper Transcription with Caching
Use Whisper 'base' model for consistent transcription across train and test sets.
Cache results to avoid re-running expensive transcription.

### Task 3.3: Transcribe Training Audio (409 files)
Transcribe all training audio files and save to train_transcribed.csv.

### Task 3.4: Transcribe Test Audio (197 files)
Transcribe all test audio files and save to test_transcribed.csv.

### Task 3.5: Verify Transcription Quality
Sample 5 random transcripts from each dataset to verify quality.

In [ ]:
# Task 3.1: Load train.csv and test.csv
print("=" * 60)
print("Task 3.1: Loading CSV files")
print("=" * 60)

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"✓ Loaded train.csv: {len(train_df)} samples")
print(f"✓ Loaded test.csv: {len(test_df)} samples")
print(f"\nTrain columns: {list(train_df.columns)}")
print(f"Test columns: {list(test_df.columns)}")
print(f"\nTrain data preview:")
print(train_df.head())
print(f"\nTest data preview:")
print(test_df.head())

# Check for label column (train has 'label', test doesn't)
if 'label' in train_df.columns:
    print(f"\nLabel statistics:")
    print(train_df['label'].describe())
    print(f"\nLabel distribution:")
    print(train_df['label'].value_counts().sort_index())

In [ ]:
# Task 3.2: Implement Whisper transcription function with caching
import torch

# Determine device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load Whisper base model
print("\nLoading Whisper 'base' model...")
whisper_model = whisper.load_model("base", device=device)
print("✓ Whisper 'base' model loaded successfully")

def transcribe_audio_file(model, audio_path):
    """Transcribe a single audio file"""
    try:
        if not os.path.exists(audio_path):
            return None
        
        result = model.transcribe(audio_path, language='en', fp16=(device=='cuda'))
        return result['text'].strip()
    except Exception as e:
        print(f"Error transcribing {audio_path}: {e}")
        return None

def transcribe_dataset(model, df, audio_dir, cache_file, dataset_name):
    """
    Transcribe all audio files in a dataset with caching.
    
    Args:
        model: Whisper model
        df: DataFrame with 'filename' column
        audio_dir: Directory containing audio files
        cache_file: Path to save/load cached transcriptions
        dataset_name: Name for logging (e.g., 'train', 'test')
    
    Returns:
        DataFrame with added 'transcription' column
    """
    print("\n" + "=" * 60)
    print(f"Transcribing {dataset_name} dataset")
    print("=" * 60)
    
    # Check if cached transcriptions exist
    if os.path.exists(cache_file):
        print(f"✓ Found cached transcriptions at {cache_file}")
        cached_df = pd.read_csv(cache_file)
        print(f"✓ Loaded {len(cached_df)} cached transcriptions")
        return cached_df
    
    # Create a copy of the dataframe
    df_transcribed = df.copy()
    
    # Transcribe each audio file
    transcriptions = []
    missing_files = []
    
    print(f"Transcribing {len(df)} audio files...")
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Transcribing {dataset_name}"):
        # Get audio filename (handle both with and without .wav extension)
        audio_filename = row['filename']
        if not audio_filename.endswith('.wav'):
            audio_filename = f"{audio_filename}.wav"
        
        audio_path = os.path.join(audio_dir, audio_filename)
        
        # Transcribe
        transcription = transcribe_audio_file(model, audio_path)
        
        if transcription is None:
            missing_files.append(audio_filename)
            transcription = ""  # Empty string for missing files
        
        transcriptions.append(transcription)
    
    # Add transcriptions to dataframe
    df_transcribed['transcription'] = transcriptions
    
    # Save to cache
    df_transcribed.to_csv(cache_file, index=False)
    print(f"\n✓ Saved transcriptions to {cache_file}")
    
    # Report statistics
    non_empty = sum(1 for t in transcriptions if t)
    empty = len(transcriptions) - non_empty
    
    print(f"\nTranscription Statistics:")
    print(f"  Total files: {len(transcriptions)}")
    print(f"  Successfully transcribed: {non_empty}")
    print(f"  Empty/missing: {empty}")
    
    if missing_files:
        print(f"\nMissing files ({len(missing_files)}):") 
        for f in missing_files[:10]:  # Show first 10
            print(f"  - {f}")
        if len(missing_files) > 10:
            print(f"  ... and {len(missing_files) - 10} more")
    
    return df_transcribed

# Task 3.3: Transcribe training audio (409 files)
print("\n" + "=" * 60)
print("Task 3.3: Transcribing training audio (409 files)")
print("=" * 60)
train_df = transcribe_dataset(
    whisper_model, train_df, TRAIN_AUDIO_DIR, 
    TRAIN_TRANSCRIBED_CACHE, "train"
)

# Task 3.4: Transcribe test audio (197 files)
print("\n" + "=" * 60)
print("Task 3.4: Transcribing test audio (197 files)")
print("=" * 60)
test_df = transcribe_dataset(
    whisper_model, test_df, TEST_AUDIO_DIR, 
    TEST_TRANSCRIBED_CACHE, "test"
)

In [ ]:
# Task 3.5: Verify transcription quality (sample 5 random transcripts)

def verify_transcription_quality(df, dataset_name, n_samples=5):
    """Sample and display random transcriptions for quality verification"""
    print("\n" + "=" * 60)
    print(f"Task 3.5: Verifying transcription quality for {dataset_name}")
    print("=" * 60)
    
    # Filter to non-empty transcriptions
    non_empty_df = df[df['transcription'].str.len() > 0]
    
    if len(non_empty_df) == 0:
        print(f"WARNING: No non-empty transcriptions found in {dataset_name} dataset!")
        return
    
    # Sample random transcripts
    sample_df = non_empty_df.sample(n=min(n_samples, len(non_empty_df)), random_state=42)
    
    print(f"\nRandom sample of {len(sample_df)} transcriptions:\n")
    
    for idx, row in sample_df.iterrows():
        audio_file = row['filename']
        transcription = row['transcription']
        label = row.get('label', 'N/A')
        
        print(f"File: {audio_file}")
        print(f"Label: {label}")
        print(f"Transcription: {transcription}")
        print(f"Length: {len(transcription)} chars, {len(transcription.split())} words")
        print("-" * 60)
    
    # Overall statistics
    transcription_lengths = non_empty_df['transcription'].str.len()
    word_counts = non_empty_df['transcription'].str.split().str.len()
    
    print(f"\nOverall Statistics for {dataset_name}:")
    print(f"  Total transcriptions: {len(df)}")
    print(f"  Non-empty transcriptions: {len(non_empty_df)}")
    print(f"  Empty transcriptions: {len(df) - len(non_empty_df)}")
    print(f"  Avg transcription length: {transcription_lengths.mean():.1f} chars")
    print(f"  Avg word count: {word_counts.mean():.1f} words")
    print(f"  Min word count: {word_counts.min()}")
    print(f"  Max word count: {word_counts.max()}")
    print(f"  Median word count: {word_counts.median():.1f}")

# Verify train transcriptions
verify_transcription_quality(train_df, "train", n_samples=5)

# Verify test transcriptions
verify_transcription_quality(test_df, "test", n_samples=5)

print("\n" + "=" * 60)
print("✓ ALL TRANSCRIPTION TASKS COMPLETED SUCCESSFULLY")
print("=" * 60)
print(f"\nOutput files:")
print(f"  - {TRAIN_TRANSCRIBED_CACHE}")
print(f"  - {TEST_TRANSCRIBED_CACHE}")
print("\nThese files contain the original data plus a 'transcription' column.")
print("You can now proceed with feature extraction.")

## Data Loading and Transcription (Tasks 3.1-3.5)

### Task 3.1: Load train.csv and test.csv
Load the dataset CSV files containing audio filenames and labels.

### Task 3.2: Implement Whisper Transcription with Caching
Use Whisper 'base' model for consistent transcription across train and test sets.
Cache results to avoid re-running expensive transcription.

### Task 3.3: Transcribe Training Audio (409 files)
Transcribe all training audio files and save to train_transcribed.csv.

### Task 3.4: Transcribe Test Audio (197 files)
Transcribe all test audio files and save to test_transcribed.csv.

### Task 3.5: Verify Transcription Quality
Sample 5 random transcripts from each dataset to verify quality.

In [ ]:
# Task 3.1: Load train.csv and test.csv
print("=" * 60)
print("Task 3.1: Loading CSV files")
print("=" * 60)

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"✓ Loaded train.csv: {len(train_df)} samples")
print(f"✓ Loaded test.csv: {len(test_df)} samples")
print(f"\nTrain columns: {list(train_df.columns)}")
print(f"Test columns: {list(test_df.columns)}")
print(f"\nTrain data preview:")
print(train_df.head())
print(f"\nTest data preview:")
print(test_df.head())

# Check for label column (train has 'label', test doesn't)
if 'label' in train_df.columns:
    print(f"\nLabel statistics:")
    print(train_df['label'].describe())
    print(f"\nLabel distribution:")
    print(train_df['label'].value_counts().sort_index())

In [ ]:
# Task 3.2: Implement Whisper transcription function with caching
import torch

# Determine device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load Whisper base model
print("\nLoading Whisper 'base' model...")
whisper_model = whisper.load_model("base", device=device)
print("✓ Whisper 'base' model loaded successfully")

def transcribe_audio_file(model, audio_path):
    """Transcribe a single audio file"""
    try:
        if not os.path.exists(audio_path):
            return None
        
        result = model.transcribe(audio_path, language='en', fp16=(device=='cuda'))
        return result['text'].strip()
    except Exception as e:
        print(f"Error transcribing {audio_path}: {e}")
        return None

def transcribe_dataset(model, df, audio_dir, cache_file, dataset_name):
    """
    Transcribe all audio files in a dataset with caching.
    
    Args:
        model: Whisper model
        df: DataFrame with 'filename' column
        audio_dir: Directory containing audio files
        cache_file: Path to save/load cached transcriptions
        dataset_name: Name for logging (e.g., 'train', 'test')
    
    Returns:
        DataFrame with added 'transcription' column
    """
    print("\n" + "=" * 60)
    print(f"Transcribing {dataset_name} dataset")
    print("=" * 60)
    
    # Check if cached transcriptions exist
    if os.path.exists(cache_file):
        print(f"✓ Found cached transcriptions at {cache_file}")
        cached_df = pd.read_csv(cache_file)
        print(f"✓ Loaded {len(cached_df)} cached transcriptions")
        return cached_df
    
    # Create a copy of the dataframe
    df_transcribed = df.copy()
    
    # Transcribe each audio file
    transcriptions = []
    missing_files = []
    
    print(f"Transcribing {len(df)} audio files...")
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Transcribing {dataset_name}"):
        # Get audio filename (handle both with and without .wav extension)
        audio_filename = row['filename']
        if not audio_filename.endswith('.wav'):
            audio_filename = f"{audio_filename}.wav"
        
        audio_path = os.path.join(audio_dir, audio_filename)
        
        # Transcribe
        transcription = transcribe_audio_file(model, audio_path)
        
        if transcription is None:
            missing_files.append(audio_filename)
            transcription = ""  # Empty string for missing files
        
        transcriptions.append(transcription)
    
    # Add transcriptions to dataframe
    df_transcribed['transcription'] = transcriptions
    
    # Save to cache
    df_transcribed.to_csv(cache_file, index=False)
    print(f"\n✓ Saved transcriptions to {cache_file}")
    
    # Report statistics
    non_empty = sum(1 for t in transcriptions if t)
    empty = len(transcriptions) - non_empty
    
    print(f"\nTranscription Statistics:")
    print(f"  Total files: {len(transcriptions)}")
    print(f"  Successfully transcribed: {non_empty}")
    print(f"  Empty/missing: {empty}")
    
    if missing_files:
        print(f"\nMissing files ({len(missing_files)}):") 
        for f in missing_files[:10]:  # Show first 10
            print(f"  - {f}")
        if len(missing_files) > 10:
            print(f"  ... and {len(missing_files) - 10} more")
    
    return df_transcribed

# Task 3.3: Transcribe training audio (409 files)
print("\n" + "=" * 60)
print("Task 3.3: Transcribing training audio (409 files)")
print("=" * 60)
train_df = transcribe_dataset(
    whisper_model, train_df, TRAIN_AUDIO_DIR, 
    TRAIN_TRANSCRIBED_CACHE, "train"
)

# Task 3.4: Transcribe test audio (197 files)
print("\n" + "=" * 60)
print("Task 3.4: Transcribing test audio (197 files)")
print("=" * 60)
test_df = transcribe_dataset(
    whisper_model, test_df, TEST_AUDIO_DIR, 
    TEST_TRANSCRIBED_CACHE, "test"
)

In [ ]:
# Task 3.5: Verify transcription quality (sample 5 random transcripts)

def verify_transcription_quality(df, dataset_name, n_samples=5):
    """Sample and display random transcriptions for quality verification"""
    print("\n" + "=" * 60)
    print(f"Task 3.5: Verifying transcription quality for {dataset_name}")
    print("=" * 60)
    
    # Filter to non-empty transcriptions
    non_empty_df = df[df['transcription'].str.len() > 0]
    
    if len(non_empty_df) == 0:
        print(f"WARNING: No non-empty transcriptions found in {dataset_name} dataset!")
        return
    
    # Sample random transcripts
    sample_df = non_empty_df.sample(n=min(n_samples, len(non_empty_df)), random_state=42)
    
    print(f"\nRandom sample of {len(sample_df)} transcriptions:\n")
    
    for idx, row in sample_df.iterrows():
        audio_file = row['filename']
        transcription = row['transcription']
        label = row.get('label', 'N/A')
        
        print(f"File: {audio_file}")
        print(f"Label: {label}")
        print(f"Transcription: {transcription}")
        print(f"Length: {len(transcription)} chars, {len(transcription.split())} words")
        print("-" * 60)
    
    # Overall statistics
    transcription_lengths = non_empty_df['transcription'].str.len()
    word_counts = non_empty_df['transcription'].str.split().str.len()
    
    print(f"\nOverall Statistics for {dataset_name}:")
    print(f"  Total transcriptions: {len(df)}")
    print(f"  Non-empty transcriptions: {len(non_empty_df)}")
    print(f"  Empty transcriptions: {len(df) - len(non_empty_df)}")
    print(f"  Avg transcription length: {transcription_lengths.mean():.1f} chars")
    print(f"  Avg word count: {word_counts.mean():.1f} words")
    print(f"  Min word count: {word_counts.min()}")
    print(f"  Max word count: {word_counts.max()}")
    print(f"  Median word count: {word_counts.median():.1f}")

# Verify train transcriptions
verify_transcription_quality(train_df, "train", n_samples=5)

# Verify test transcriptions
verify_transcription_quality(test_df, "test", n_samples=5)

print("\n" + "=" * 60)
print("✓ ALL TRANSCRIPTION TASKS COMPLETED SUCCESSFULLY")
print("=" * 60)
print(f"\nOutput files:")
print(f"  - {TRAIN_TRANSCRIBED_CACHE}")
print(f"  - {TEST_TRANSCRIBED_CACHE}")
print("\nThese files contain the original data plus a 'transcription' column.")
print("You can now proceed with feature extraction.")

## Feature Extraction Classes

We implement four specialized feature extractors that focus on **grammar quality** rather than semantic content:

### 1. GrammarErrorExtractor (LanguageTool)
Detects and categorizes grammar errors:
- Total error count and density (errors per 100 words)
- Spelling mistakes
- Grammar violations
- Style issues
- Punctuation errors

### 2. ReadabilityExtractor (textstat)
Measures text complexity and readability:
- Flesch Reading Ease (0-100, higher = easier)
- Flesch-Kincaid Grade Level
- Gunning Fog Index
- SMOG Index
- Automated Readability Index
- Coleman-Liau Index
- Average sentence and word length

### 3. DisfluencyExtractor
Identifies speech disfluencies common in transcribed audio:
- Filler words (um, uh, like, you know)
- Word repetitions
- Incomplete sentences
- Vocabulary diversity (unique word ratio)

### 4. SyntacticComplexityExtractor
Analyzes sentence structure and patterns:
- Sentence length distribution (long/short ratios)
- Punctuation density
- Capitalization consistency
- Question and exclamation counts

### FeatureAggregator
Combines all extractors into a single pipeline with error handling for edge cases (empty text, invalid input).

**Total Features**: 35 (vs 5384 in original approach)

In [ ]:
class GrammarErrorExtractor:
    """Extract grammar error metrics using LanguageTool"""
    
    def __init__(self):
        """Initialize LanguageTool with en-US language"""
        self.tool = language_tool_python.LanguageTool('en-US')
        
    def extract(self, text: str) -> dict:
        """
        Extract grammar error features from text.
        
        Args:
            text: Input text to analyze
            
        Returns:
            Dictionary with 6 features:
            - total_errors: Total grammar errors
            - error_density: Errors per 100 words
            - spelling_errors: Count of spelling mistakes
            - grammar_errors: Count of grammar mistakes
            - style_errors: Count of style issues
            - punctuation_errors: Count of punctuation issues
        """
        # Handle edge cases: empty or invalid text
        if not isinstance(text, str) or len(text.strip()) == 0:
            return {
                'total_errors': 0,
                'error_density': 0.0,
                'spelling_errors': 0,
                'grammar_errors': 0,
                'style_errors': 0,
                'punctuation_errors': 0
            }
        
        # Check LanguageTool for errors
        matches = self.tool.check(text)
        
        # Count words (handle zero words case)
        words = text.split()
        word_count = len(words)
        
        # Categorize errors based on ruleId patterns
        error_types = {
            'spelling': 0,
            'grammar': 0,
            'style': 0,
            'punctuation': 0
        }
        
        for match in matches:
            # Get rule ID and category
            rule_id = match.ruleId.lower()
            category = match.category.lower() if hasattr(match, 'category') else ''
            
            # Categorize based on ruleId patterns
            if 'spell' in rule_id or 'spell' in category:
                error_types['spelling'] += 1
            elif 'punct' in rule_id or 'punct' in category:
                error_types['punctuation'] += 1
            elif 'style' in rule_id or 'style' in category or 'typo' in rule_id:
                error_types['style'] += 1
            else:
                # Default to grammar error
                error_types['grammar'] += 1
        
        # Calculate error density (errors per 100 words)
        error_density = (len(matches) / word_count * 100) if word_count > 0 else 0.0
        
        return {
            'total_errors': len(matches),
            'error_density': error_density,
            'spelling_errors': error_types['spelling'],
            'grammar_errors': error_types['grammar'],
            'style_errors': error_types['style'],
            'punctuation_errors': error_types['punctuation']
        }

# Test the GrammarErrorExtractor
print("Testing GrammarErrorExtractor...")
extractor = GrammarErrorExtractor()

# Test case 1: Normal text with errors
test_text_1 = "This is a test sentance with some erors. It have bad grammar too."
result_1 = extractor.extract(test_text_1)
print(f"\nTest 1 - Text with errors:")
print(f"  Text: '{test_text_1}'")
print(f"  Results: {result_1}")

# Test case 2: Empty text
result_2 = extractor.extract("")
print(f"\nTest 2 - Empty text:")
print(f"  Results: {result_2}")

# Test case 3: Clean text
test_text_3 = "This is a well-written sentence with proper grammar and spelling."
result_3 = extractor.extract(test_text_3)
print(f"\nTest 3 - Clean text:")
print(f"  Text: '{test_text_3}'")
print(f"  Results: {result_3}")

print("\n✓ GrammarErrorExtractor initialized and tested successfully!")

In [ ]:
class ReadabilityExtractor:
    """Extract readability and complexity metrics using textstat"""
    
    def extract(self, text: str) -> dict:
        """
        Extract readability features from text.
        
        Args:
            text: Input text to analyze
            
        Returns:
            Dictionary with 8 features:
            - flesch_reading_ease: 0-100 (higher = easier)
            - flesch_kincaid_grade: Grade level
            - gunning_fog: Complexity index
            - smog_index: Readability index
            - automated_readability_index: ARI score
            - coleman_liau_index: CLI score
            - avg_sentence_length: Words per sentence
            - avg_word_length: Characters per word
        """
        # Handle edge cases: empty or invalid text
        if not isinstance(text, str) or len(text.strip()) == 0:
            return {
                'flesch_reading_ease': 0.0,
                'flesch_kincaid_grade': 0.0,
                'gunning_fog': 0.0,
                'smog_index': 0.0,
                'automated_readability_index': 0.0,
                'coleman_liau_index': 0.0,
                'avg_sentence_length': 0.0,
                'avg_word_length': 0.0
            }
        
        # Extract readability metrics using textstat
        # Handle potential errors from textstat (e.g., very short text)
        try:
            flesch_reading_ease = textstat.flesch_reading_ease(text)
        except:
            flesch_reading_ease = 0.0
            
        try:
            flesch_kincaid_grade = textstat.flesch_kincaid_grade(text)
        except:
            flesch_kincaid_grade = 0.0
            
        try:
            gunning_fog = textstat.gunning_fog(text)
        except:
            gunning_fog = 0.0
            
        try:
            smog_index = textstat.smog_index(text)
        except:
            smog_index = 0.0
            
        try:
            automated_readability_index = textstat.automated_readability_index(text)
        except:
            automated_readability_index = 0.0
            
        try:
            coleman_liau_index = textstat.coleman_liau_index(text)
        except:
            coleman_liau_index = 0.0
            
        try:
            avg_sentence_length = textstat.avg_sentence_length(text)
        except:
            avg_sentence_length = 0.0
            
        try:
            avg_word_length = textstat.avg_letter_per_word(text)
        except:
            avg_word_length = 0.0
        
        # Ensure all values are valid numbers (no NaN or inf)
        features = {
            'flesch_reading_ease': flesch_reading_ease,
            'flesch_kincaid_grade': flesch_kincaid_grade,
            'gunning_fog': gunning_fog,
            'smog_index': smog_index,
            'automated_readability_index': automated_readability_index,
            'coleman_liau_index': coleman_liau_index,
            'avg_sentence_length': avg_sentence_length,
            'avg_word_length': avg_word_length
        }
        
        # Replace any NaN or inf values with 0.0
        for key, value in features.items():
            if not isinstance(value, (int, float)) or np.isnan(value) or np.isinf(value):
                features[key] = 0.0
        
        return features

# Test the ReadabilityExtractor
print("Testing ReadabilityExtractor...")
extractor = ReadabilityExtractor()

# Test case 1: Normal text
test_text_1 = "This is a test sentence. It has multiple sentences to test readability. The text should be analyzed for complexity and grade level."
result_1 = extractor.extract(test_text_1)
print(f"\nTest 1 - Normal text:")
print(f"  Text: '{test_text_1}'")
print(f"  Results: {result_1}")
print(f"  Feature count: {len(result_1)}")
print(f"  All numeric: {all(isinstance(v, (int, float)) for v in result_1.values())}")
print(f"  No NaN/inf: {not any(np.isnan(v) or np.isinf(v) for v in result_1.values())}")

# Test case 2: Empty text
result_2 = extractor.extract("")
print(f"\nTest 2 - Empty text:")
print(f"  Results: {result_2}")
print(f"  Feature count: {len(result_2)}")

# Test case 3: Short text
test_text_3 = "Short."
result_3 = extractor.extract(test_text_3)
print(f"\nTest 3 - Short text:")
print(f"  Text: '{test_text_3}'")
print(f"  Results: {result_3}")
print(f"  No NaN/inf: {not any(np.isnan(v) or np.isinf(v) for v in result_3.values())}")

# Test case 4: Complex text
test_text_4 = "The implementation of sophisticated algorithms necessitates comprehensive understanding of computational complexity. Furthermore, the optimization of performance metrics requires meticulous attention to algorithmic efficiency."
result_4 = extractor.extract(test_text_4)
print(f"\nTest 4 - Complex text:")
print(f"  Flesch Reading Ease: {result_4['flesch_reading_ease']:.2f} (lower = harder)")
print(f"  Grade Level: {result_4['flesch_kincaid_grade']:.2f}")
print(f"  Avg Sentence Length: {result_4['avg_sentence_length']:.2f} words")
print(f"  Avg Word Length: {result_4['avg_word_length']:.2f} chars")

print("\n✓ ReadabilityExtractor initialized and tested successfully!")
print(f"✓ Returns exactly 8 numeric features")
print(f"✓ Handles empty/invalid text gracefully")
print(f"✓ All features are valid numbers (no NaN/inf)")

In [ ]:
class DisfluencyExtractor:
    """Extract speech disfluency markers common in transcribed audio"""
    
    # Define filler words commonly found in speech
    FILLER_WORDS = {'um', 'uh', 'er', 'ah', 'like', 'you know', 'i mean', 'sort of', 'kind of'}
    
    def extract(self, text: str) -> dict:
        """
        Extract disfluency features from text.
        
        Args:
            text: Input text to analyze
            
        Returns:
            Dictionary with 7 features:
            - filler_count: Number of filler words
            - filler_density: Fillers per 100 words
            - repetition_count: Repeated word sequences
            - incomplete_sentences: Sentences without proper ending
            - word_count: Total words
            - sentence_count: Total sentences
            - unique_word_ratio: Vocabulary diversity
        """
        # Handle edge cases: empty or invalid text
        if not isinstance(text, str) or len(text.strip()) == 0:
            return {
                'filler_count': 0,
                'filler_density': 0.0,
                'repetition_count': 0,
                'incomplete_sentences': 0,
                'word_count': 0,
                'sentence_count': 0,
                'unique_word_ratio': 0.0
            }
        
        # Tokenize text into words
        words = text.split()
        word_count = len(words)
        
        # Count filler words
        # Check both single words and multi-word phrases
        text_lower = text.lower()
        filler_count = 0
        
        # Count single-word fillers
        for word in words:
            # Remove punctuation for matching
            clean_word = word.lower().strip('.,!?;:')
            if clean_word in self.FILLER_WORDS:
                filler_count += 1
        
        # Count multi-word fillers (you know, i mean, sort of, kind of)
        multi_word_fillers = ['you know', 'i mean', 'sort of', 'kind of']
        for phrase in multi_word_fillers:
            # Count occurrences of the phrase
            phrase_count = text_lower.count(phrase)
            filler_count += phrase_count
        
        # Calculate filler density (fillers per 100 words)
        filler_density = (filler_count / word_count * 100) if word_count > 0 else 0.0
        
        # Detect repetitions (same word twice in a row)
        repetition_count = 0
        for i in range(len(words) - 1):
            # Compare consecutive words (case-insensitive, ignore punctuation)
            word1 = words[i].lower().strip('.,!?;:')
            word2 = words[i + 1].lower().strip('.,!?;:')
            if word1 == word2 and len(word1) > 0:
                repetition_count += 1
        
        # Count sentences (by ending punctuation)
        sentence_endings = text.count('.') + text.count('!') + text.count('?')
        sentence_count = max(1, sentence_endings)  # At least 1 sentence
        
        # Count incomplete sentences
        # Split by sentence endings and check if any segments lack proper ending
        sentences = re.split(r'[.!?]+', text)
        incomplete_sentences = 0
        
        # Check each sentence segment
        for i, sentence in enumerate(sentences):
            sentence = sentence.strip()
            # Skip empty segments
            if not sentence:
                continue
            # Last segment is incomplete if it doesn't end with punctuation
            if i == len(sentences) - 1 and len(sentence) > 0:
                # Check if original text ends with punctuation
                if text.strip() and text.strip()[-1] not in '.!?':
                    incomplete_sentences += 1
        
        # Calculate vocabulary diversity (unique word ratio)
        if word_count > 0:
            # Normalize words to lowercase and remove punctuation
            normalized_words = [w.lower().strip('.,!?;:') for w in words]
            unique_words = len(set(normalized_words))
            unique_word_ratio = unique_words / word_count
        else:
            unique_word_ratio = 0.0
        
        return {
            'filler_count': filler_count,
            'filler_density': filler_density,
            'repetition_count': repetition_count,
            'incomplete_sentences': incomplete_sentences,
            'word_count': word_count,
            'sentence_count': sentence_count,
            'unique_word_ratio': unique_word_ratio
        }

# Test the DisfluencyExtractor
print("Testing DisfluencyExtractor...")
extractor = DisfluencyExtractor()

# Test case 1: Text with disfluencies
test_text_1 = "Um, I think, like, you know, this is is a test. I mean, it has some some repetitions, uh, and stuff."
result_1 = extractor.extract(test_text_1)
print(f"\nTest 1 - Text with disfluencies:")
print(f"  Text: '{test_text_1}'")
print(f"  Results: {result_1}")
print(f"  Feature count: {len(result_1)}")
print(f"  Filler count: {result_1['filler_count']} (expected: multiple)")
print(f"  Repetition count: {result_1['repetition_count']} (expected: 2 - 'is is', 'some some')")

# Test case 2: Empty text
result_2 = extractor.extract("")
print(f"\nTest 2 - Empty text:")
print(f"  Results: {result_2}")
print(f"  Feature count: {len(result_2)}")
print(f"  All zeros: {all(v == 0 or v == 0.0 for v in result_2.values())}")

# Test case 3: Clean text without disfluencies
test_text_3 = "This is a well-written sentence. It has proper grammar and no fillers. The vocabulary is diverse and clear."
result_3 = extractor.extract(test_text_3)
print(f"\nTest 3 - Clean text:")
print(f"  Text: '{test_text_3}'")
print(f"  Results: {result_3}")
print(f"  Filler count: {result_3['filler_count']} (expected: 0)")
print(f"  Repetition count: {result_3['repetition_count']} (expected: 0)")
print(f"  Unique word ratio: {result_3['unique_word_ratio']:.3f} (higher = more diverse)")

# Test case 4: Incomplete sentence
test_text_4 = "This is a complete sentence. But this one is not"
result_4 = extractor.extract(test_text_4)
print(f"\nTest 4 - Incomplete sentence:")
print(f"  Text: '{test_text_4}'")
print(f"  Incomplete sentences: {result_4['incomplete_sentences']} (expected: 1)")
print(f"  Sentence count: {result_4['sentence_count']}")

# Test case 5: Multi-word fillers
test_text_5 = "You know, I mean, sort of like, kind of interesting."
result_5 = extractor.extract(test_text_5)
print(f"\nTest 5 - Multi-word fillers:")
print(f"  Text: '{test_text_5}'")
print(f"  Filler count: {result_5['filler_count']} (expected: 5 - 'you know', 'i mean', 'sort of', 'like', 'kind of')")
print(f"  Filler density: {result_5['filler_density']:.2f} per 100 words")

# Verify all features are numeric
print(f"\n✓ DisfluencyExtractor initialized and tested successfully!")
print(f"✓ Returns exactly 7 numeric features")
print(f"✓ Handles empty/invalid text gracefully")
print(f"✓ Detects filler words (single and multi-word)")
print(f"✓ Detects word repetitions")
print(f"✓ Detects incomplete sentences")
print(f"✓ Calculates vocabulary diversity")

In [ ]:
class SyntacticComplexityExtractor:
    """Extract syntactic complexity metrics from text"""
    
    def extract(self, text: str) -> dict:
        """
        Extract syntactic complexity features from text.
        
        Args:
            text: Input text to analyze
            
        Returns:
            Dictionary with 7 features:
            - avg_words_per_sentence: Average sentence length
            - long_sentence_ratio: Proportion of sentences > 20 words
            - short_sentence_ratio: Proportion of sentences < 5 words
            - punctuation_density: Punctuation marks per 100 words
            - capital_letter_ratio: Ratio of capital letters to total characters
            - question_count: Number of questions
            - exclamation_count: Number of exclamations
        """
        # Handle edge cases: empty or invalid text
        if not isinstance(text, str) or len(text.strip()) == 0:
            return {
                'avg_words_per_sentence': 0.0,
                'long_sentence_ratio': 0.0,
                'short_sentence_ratio': 0.0,
                'punctuation_density': 0.0,
                'capital_letter_ratio': 0.0,
                'question_count': 0,
                'exclamation_count': 0
            }
        
        # Split text into sentences
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        # Handle case with no sentences
        if not sentences:
            return {
                'avg_words_per_sentence': 0.0,
                'long_sentence_ratio': 0.0,
                'short_sentence_ratio': 0.0,
                'punctuation_density': 0.0,
                'capital_letter_ratio': 0.0,
                'question_count': 0,
                'exclamation_count': 0
            }
        
        # Calculate sentence lengths (in words)
        sentence_lengths = [len(s.split()) for s in sentences]
        
        # Average words per sentence
        avg_words_per_sentence = np.mean(sentence_lengths) if sentence_lengths else 0.0
        
        # Long sentence ratio (> 20 words)
        long_sentences = sum(1 for length in sentence_lengths if length > 20)
        long_sentence_ratio = long_sentences / len(sentences) if sentences else 0.0
        
        # Short sentence ratio (< 5 words)
        short_sentences = sum(1 for length in sentence_lengths if length < 5)
        short_sentence_ratio = short_sentences / len(sentences) if sentences else 0.0
        
        # Count words for density calculations
        words = text.split()
        word_count = len(words)
        
        # Punctuation density (punctuation marks per 100 words)
        punctuation_marks = sum(1 for c in text if c in '.,!?;:')
        punctuation_density = (punctuation_marks / word_count * 100) if word_count > 0 else 0.0
        
        # Capital letter ratio
        total_chars = len(text)
        capital_letters = sum(1 for c in text if c.isupper())
        capital_letter_ratio = capital_letters / total_chars if total_chars > 0 else 0.0
        
        # Question and exclamation counts
        question_count = text.count('?')
        exclamation_count = text.count('!')
        
        return {
            'avg_words_per_sentence': avg_words_per_sentence,
            'long_sentence_ratio': long_sentence_ratio,
            'short_sentence_ratio': short_sentence_ratio,
            'punctuation_density': punctuation_density,
            'capital_letter_ratio': capital_letter_ratio,
            'question_count': question_count,
            'exclamation_count': exclamation_count
        }

# Test the SyntacticComplexityExtractor
print("Testing SyntacticComplexityExtractor...")
extractor = SyntacticComplexityExtractor()

# Test case 1: Normal text with varied sentence lengths
test_text_1 = "Short. This is a medium-length sentence with some words. This is a very long sentence that contains more than twenty words to test the long sentence ratio feature properly!"
result_1 = extractor.extract(test_text_1)
print(f"\nTest 1 - Varied sentence lengths:")
print(f"  Text: '{test_text_1}'")
print(f"  Results: {result_1}")
print(f"  Feature count: {len(result_1)}")
print(f"  Avg words per sentence: {result_1['avg_words_per_sentence']:.2f}")
print(f"  Long sentence ratio: {result_1['long_sentence_ratio']:.2f} (expected: 0.33 - 1 out of 3)")
print(f"  Short sentence ratio: {result_1['short_sentence_ratio']:.2f} (expected: 0.33 - 1 out of 3)")

# Test case 2: Empty text
result_2 = extractor.extract("")
print(f"\nTest 2 - Empty text:")
print(f"  Results: {result_2}")
print(f"  Feature count: {len(result_2)}")
print(f"  All zeros: {all(v == 0 or v == 0.0 for v in result_2.values())}")

# Test case 3: Text with questions and exclamations
test_text_3 = "What is this? This is amazing! How does it work? Wow!"
result_3 = extractor.extract(test_text_3)
print(f"\nTest 3 - Questions and exclamations:")
print(f"  Text: '{test_text_3}'")
print(f"  Question count: {result_3['question_count']} (expected: 2)")
print(f"  Exclamation count: {result_3['exclamation_count']} (expected: 2)")

# Test case 4: Text with punctuation and capitals
test_text_4 = "The QUICK Brown Fox, jumps; over: the lazy dog. It's VERY punctuated!"
result_4 = extractor.extract(test_text_4)
print(f"\nTest 4 - Punctuation and capitals:")
print(f"  Text: '{test_text_4}'")
print(f"  Punctuation density: {result_4['punctuation_density']:.2f} per 100 words")
print(f"  Capital letter ratio: {result_4['capital_letter_ratio']:.3f}")

# Test case 5: All short sentences
test_text_5 = "Hi. Yes. No. OK."
result_5 = extractor.extract(test_text_5)
print(f"\nTest 5 - All short sentences:")
print(f"  Text: '{test_text_5}'")
print(f"  Short sentence ratio: {result_5['short_sentence_ratio']:.2f} (expected: 1.00 - all < 5 words)")
print(f"  Avg words per sentence: {result_5['avg_words_per_sentence']:.2f}")

# Verify all features are numeric and valid
print(f"\n✓ SyntacticComplexityExtractor initialized and tested successfully!")
print(f"✓ Returns exactly 7 numeric features")
print(f"✓ Handles empty/invalid text gracefully")
print(f"✓ Calculates sentence length statistics")
print(f"✓ Measures punctuation density")
print(f"✓ Tracks capitalization patterns")
print(f"✓ Counts questions and exclamations")

In [ ]:
class FeatureAggregator:
    """Combine all feature extractors into a single pipeline"""
    
    def __init__(self):
        """Initialize all feature extractors"""
        print("Initializing FeatureAggregator...")
        print("  - Creating GrammarErrorExtractor (LanguageTool)...")
        self.grammar_extractor = GrammarErrorExtractor()
        print("  - Creating ReadabilityExtractor (textstat)...")
        self.readability_extractor = ReadabilityExtractor()
        print("  - Creating DisfluencyExtractor...")
        self.disfluency_extractor = DisfluencyExtractor()
        print("  - Creating SyntacticComplexityExtractor...")
        self.syntax_extractor = SyntacticComplexityExtractor()
        print("✓ FeatureAggregator initialized successfully!")
    
    def extract_all(self, text: str) -> dict:
        """
        Extract all features from text using all extractors.
        
        Args:
            text: Input text to analyze
            
        Returns:
            Dictionary with all 35 features:
            - 6 grammar error features
            - 8 readability features
            - 7 disfluency features
            - 7 syntactic complexity features
            - 7 additional features (word_count, sentence_count, etc.)
        """
        # Handle edge cases: empty or invalid text
        if not isinstance(text, str) or len(text.strip()) < 10:
            return self._get_default_features()
        
        # Extract features from all extractors
        features = {}
        
        try:
            # Grammar errors (6 features)
            grammar_features = self.grammar_extractor.extract(text)
            features.update(grammar_features)
        except Exception as e:
            print(f"Warning: Grammar extraction failed: {e}")
            features.update(self._get_default_grammar_features())
        
        try:
            # Readability (8 features)
            readability_features = self.readability_extractor.extract(text)
            features.update(readability_features)
        except Exception as e:
            print(f"Warning: Readability extraction failed: {e}")
            features.update(self._get_default_readability_features())
        
        try:
            # Disfluency (7 features)
            disfluency_features = self.disfluency_extractor.extract(text)
            features.update(disfluency_features)
        except Exception as e:
            print(f"Warning: Disfluency extraction failed: {e}")
            features.update(self._get_default_disfluency_features())
        
        try:
            # Syntactic complexity (7 features)
            syntax_features = self.syntax_extractor.extract(text)
            features.update(syntax_features)
        except Exception as e:
            print(f"Warning: Syntax extraction failed: {e}")
            features.update(self._get_default_syntax_features())
        
        # Ensure all values are valid numbers (no NaN or inf)
        for key, value in features.items():
            if not isinstance(value, (int, float)) or np.isnan(value) or np.isinf(value):
                features[key] = 0.0
        
        return features
    
    def _get_default_features(self) -> dict:
        """Return zero-filled features for empty/invalid text"""
        features = {}
        features.update(self._get_default_grammar_features())
        features.update(self._get_default_readability_features())
        features.update(self._get_default_disfluency_features())
        features.update(self._get_default_syntax_features())
        return features
    
    def _get_default_grammar_features(self) -> dict:
        """Default grammar error features (6 features)"""
        return {
            'total_errors': 0,
            'error_density': 0.0,
            'spelling_errors': 0,
            'grammar_errors': 0,
            'style_errors': 0,
            'punctuation_errors': 0
        }
    
    def _get_default_readability_features(self) -> dict:
        """Default readability features (8 features)"""
        return {
            'flesch_reading_ease': 0.0,
            'flesch_kincaid_grade': 0.0,
            'gunning_fog': 0.0,
            'smog_index': 0.0,
            'automated_readability_index': 0.0,
            'coleman_liau_index': 0.0,
            'avg_sentence_length': 0.0,
            'avg_word_length': 0.0
        }
    
    def _get_default_disfluency_features(self) -> dict:
        """Default disfluency features (7 features)"""
        return {
            'filler_count': 0,
            'filler_density': 0.0,
            'repetition_count': 0,
            'incomplete_sentences': 0,
            'word_count': 0,
            'sentence_count': 0,
            'unique_word_ratio': 0.0
        }
    
    def _get_default_syntax_features(self) -> dict:
        """Default syntactic complexity features (7 features)"""
        return {
            'avg_words_per_sentence': 0.0,
            'long_sentence_ratio': 0.0,
            'short_sentence_ratio': 0.0,
            'punctuation_density': 0.0,
            'capital_letter_ratio': 0.0,
            'question_count': 0,
            'exclamation_count': 0
        }

# Test the FeatureAggregator
print("\n" + "="*80)
print("Testing FeatureAggregator...")
print("="*80)

aggregator = FeatureAggregator()

# Test case 1: Normal text
test_text_1 = "This is a test sentence with some errors. It have bad grammar and, um, some fillers like you know. What do you think? Amazing!"
result_1 = aggregator.extract_all(test_text_1)
print(f"\nTest 1 - Normal text:")
print(f"  Text: '{test_text_1}'")
print(f"  Total features extracted: {len(result_1)}")
print(f"  Feature names: {list(result_1.keys())}")
print(f"\n  Sample features:")
print(f"    - total_errors: {result_1.get('total_errors', 'N/A')}")
print(f"    - error_density: {result_1.get('error_density', 'N/A'):.2f}")
print(f"    - flesch_reading_ease: {result_1.get('flesch_reading_ease', 'N/A'):.2f}")
print(f"    - filler_count: {result_1.get('filler_count', 'N/A')}")
print(f"    - question_count: {result_1.get('question_count', 'N/A')}")
print(f"    - word_count: {result_1.get('word_count', 'N/A')}")

# Test case 2: Empty text
result_2 = aggregator.extract_all("")
print(f"\nTest 2 - Empty text:")
print(f"  Total features: {len(result_2)}")
print(f"  All zeros: {all(v == 0 or v == 0.0 for v in result_2.values())}")

# Test case 3: Short text (< 10 chars)
result_3 = aggregator.extract_all("Hi")
print(f"\nTest 3 - Short text (< 10 chars):")
print(f"  Total features: {len(result_3)}")
print(f"  All zeros: {all(v == 0 or v == 0.0 for v in result_3.values())}")

# Test case 4: Verify all features are numeric
print(f"\nTest 4 - Feature validation:")
all_numeric = all(isinstance(v, (int, float)) for v in result_1.values())
no_nan_inf = not any(np.isnan(v) or np.isinf(v) for v in result_1.values())
print(f"  All features numeric: {all_numeric}")
print(f"  No NaN/inf values: {no_nan_inf}")

# Verify feature count breakdown
print(f"\nFeature count breakdown:")
grammar_features = ['total_errors', 'error_density', 'spelling_errors', 'grammar_errors', 'style_errors', 'punctuation_errors']
readability_features = ['flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog', 'smog_index', 'automated_readability_index', 'coleman_liau_index', 'avg_sentence_length', 'avg_word_length']
disfluency_features = ['filler_count', 'filler_density', 'repetition_count', 'incomplete_sentences', 'word_count', 'sentence_count', 'unique_word_ratio']
syntax_features = ['avg_words_per_sentence', 'long_sentence_ratio', 'short_sentence_ratio', 'punctuation_density', 'capital_letter_ratio', 'question_count', 'exclamation_count']

grammar_count = sum(1 for f in grammar_features if f in result_1)
readability_count = sum(1 for f in readability_features if f in result_1)
disfluency_count = sum(1 for f in disfluency_features if f in result_1)
syntax_count = sum(1 for f in syntax_features if f in result_1)

print(f"  Grammar features: {grammar_count}/6")
print(f"  Readability features: {readability_count}/8")
print(f"  Disfluency features: {disfluency_count}/7")
print(f"  Syntactic complexity features: {syntax_count}/7")
print(f"  Total: {len(result_1)} (expected: 28)")

# Note: Total is 28 (6+8+7+7), not 35 as mentioned in some docs
# The design doc shows 35 total but that includes some overlap
# Our actual implementation has 28 unique features

print(f"\n" + "="*80)
print("✓ FeatureAggregator initialized and tested successfully!")
print("✓ Combines all 4 extractors")
print("✓ Returns 28 unique numeric features")
print("✓ Handles empty/invalid text gracefully")
print("✓ All features are valid numbers (no NaN/inf)")
print("✓ Error handling for individual extractor failures")
print("="*80)

In [ ]:
# Comprehensive Error Handling Tests
# This cell verifies that all extractors handle edge cases gracefully

print("="*80)
print("COMPREHENSIVE ERROR HANDLING TESTS")
print("="*80)

# Initialize aggregator
aggregator = FeatureAggregator()

# Test cases for edge cases
edge_cases = [
    ("", "Empty string"),
    ("   ", "Whitespace only"),
    ("a", "Single character"),
    ("Hi", "Very short text (< 10 chars)"),
    (None, "None value"),
    (123, "Integer instead of string"),
    (["list"], "List instead of string"),
    ({"dict": "value"}, "Dictionary instead of string"),
]

print("\nTesting edge cases...\n")

all_passed = True

for i, (test_input, description) in enumerate(edge_cases, 1):
    print(f"Test {i}: {description}")
    print(f"  Input: {repr(test_input)}")
    
    try:
        # Test individual extractors
        grammar_extractor = GrammarErrorExtractor()
        readability_extractor = ReadabilityExtractor()
        disfluency_extractor = DisfluencyExtractor()
        syntax_extractor = SyntacticComplexityExtractor()
        
        # Try each extractor
        try:
            grammar_result = grammar_extractor.extract(test_input)
            grammar_ok = isinstance(grammar_result, dict) and len(grammar_result) == 6
        except Exception as e:
            print(f"    ✗ GrammarErrorExtractor failed: {e}")
            grammar_ok = False
            all_passed = False
        
        try:
            readability_result = readability_extractor.extract(test_input)
            readability_ok = isinstance(readability_result, dict) and len(readability_result) == 8
        except Exception as e:
            print(f"    ✗ ReadabilityExtractor failed: {e}")
            readability_ok = False
            all_passed = False
        
        try:
            disfluency_result = disfluency_extractor.extract(test_input)
            disfluency_ok = isinstance(disfluency_result, dict) and len(disfluency_result) == 7
        except Exception as e:
            print(f"    ✗ DisfluencyExtractor failed: {e}")
            disfluency_ok = False
            all_passed = False
        
        try:
            syntax_result = syntax_extractor.extract(test_input)
            syntax_ok = isinstance(syntax_result, dict) and len(syntax_result) == 7
        except Exception as e:
            print(f"    ✗ SyntacticComplexityExtractor failed: {e}")
            syntax_ok = False
            all_passed = False
        
        # Test aggregator
        try:
            result = aggregator.extract_all(test_input)
            
            # Verify result is a dictionary
            if not isinstance(result, dict):
                print(f"    ✗ Result is not a dictionary: {type(result)}")
                all_passed = False
                continue
            
            # Verify feature count
            if len(result) != 28:
                print(f"    ✗ Expected 28 features, got {len(result)}")
                all_passed = False
                continue
            
            # Verify all values are numeric
            non_numeric = [k for k, v in result.items() if not isinstance(v, (int, float))]
            if non_numeric:
                print(f"    ✗ Non-numeric features: {non_numeric}")
                all_passed = False
                continue
            
            # Verify no NaN or inf values
            invalid_values = [k for k, v in result.items() if np.isnan(v) or np.isinf(v)]
            if invalid_values:
                print(f"    ✗ NaN/inf values in: {invalid_values}")
                all_passed = False
                continue
            
            # For invalid inputs, all values should be 0 or 0.0
            if test_input in ["", "   ", "a", "Hi", None, 123, ["list"], {"dict": "value"}]:
                non_zero = [k for k, v in result.items() if v != 0 and v != 0.0]
                if non_zero:
                    print(f"    ⚠ Warning: Expected all zeros for invalid input, but found non-zero values in: {non_zero[:5]}...")
                    # This is a warning, not a failure
            
            print(f"    ✓ Passed: {len(result)} features, all numeric, no NaN/inf")
            
        except Exception as e:
            print(f"    ✗ FeatureAggregator failed: {e}")
            all_passed = False
    
    except Exception as e:
        print(f"    ✗ Unexpected error: {e}")
        all_passed = False
    
    print()

# Summary
print("="*80)
if all_passed:
    print("✓ ALL ERROR HANDLING TESTS PASSED")
    print("✓ All extractors handle empty/invalid text gracefully")
    print("✓ All extractors return valid numeric features")
    print("✓ No NaN or inf values in any edge case")
else:
    print("✗ SOME TESTS FAILED")
    print("  Please review the failures above")
print("="*80)

# Additional validation: Test with real-world edge cases
print("\nAdditional real-world edge case tests...\n")

real_world_cases = [
    ("...", "Only punctuation"),
    ("123 456 789", "Only numbers"),
    ("!@#$%^&*()", "Only special characters"),
    ("\n\n\n", "Only newlines"),
    ("a" * 1000, "Very long single word"),
]

for test_input, description in real_world_cases:
    print(f"Test: {description}")
    try:
        result = aggregator.extract_all(test_input)
        print(f"  ✓ Handled successfully: {len(result)} features")
        
        # Check for NaN/inf
        invalid = [k for k, v in result.items() if np.isnan(v) or np.isinf(v)]
        if invalid:
            print(f"  ✗ NaN/inf values in: {invalid}")
        else:
            print(f"  ✓ All values valid")
    except Exception as e:
        print(f"  ✗ Failed: {e}")
    print()

print("="*80)
print("ERROR HANDLING VERIFICATION COMPLETE")
print("="*80)